# Official EXO-200 v1 Transformer Training

Runs the standardized 3-tokenization × 3-position-encoding classification matrix. EXOBench remains authoritative for loading, preprocessing, run-level splitting, optimization, checkpoint selection, and test evaluation.

In [1]:
from pathlib import Path
import json
import math
import os
import sys
import time

import pandas as pd
import torch

configured_root = os.environ.get('EXO200_TRANSFORMER_PROJECT_ROOT')
candidate_roots = []
if configured_root:
    candidate_roots.append(Path(configured_root).expanduser())
candidate_roots.extend([
    Path.cwd() / 'exo200_detector',
    Path.cwd(),
    Path.cwd().parent / 'exo200_detector',
    Path.cwd().parent,
    Path.cwd().parent.parent / 'exo200_detector',
])
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in candidate_roots
     if (candidate / 'exo_transformer').is_dir()
     and (candidate / 'exobench').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate exo200_detector; set EXO200_TRANSFORMER_PROJECT_ROOT.'
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from exobench import (
    DataConfig, TrainingConfig, evaluate_model, prepare_dataset,
    set_seed, train_model,
)
from exo_transformer import (
    EXOTransformerClassifier, TokenizationConfig, validate_split_manifest,
)

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('EXOBench:', Path(sys.modules['exobench'].__file__).resolve())
print('Transformer:', Path(sys.modules['exo_transformer'].__file__).resolve())

PyTorch: 2.11.0+cu128
CUDA available: True
EXOBench: /home/klz/Data/zeronu_benchmark/Transformer_Approach/exo200_detector/exobench/__init__.py
Transformer: /home/klz/Data/zeronu_benchmark/Transformer_Approach/exo200_detector/exo_transformer/__init__.py


In [2]:
DATA_ROOT = "/home/klz/Data/zeronu_benchmark/EXO-200"
OUTPUT_ROOT = Path(os.environ.get(
    'EXO200_OUTPUT_ROOT',
    str(PROJECT_ROOT / 'results' / 'transformer_official_v1'),
)).expanduser()
SUMMARY_PATH = OUTPUT_ROOT / 'transformer_results.csv'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

data_config = DataConfig(
    data_root=DATA_ROOT,
    validation_fraction=0.10,
    baseline_samples=200,
    classification_amplitude_normalization=True,
    seed=42,
)
training_config = TrainingConfig()  # Shared collaborator defaults.

# Add a run ID only when intentionally treating it as already complete.
COMPLETED_OFFICIAL_RUNS = set()

print('Dataset:', DATA_ROOT)
print('Outputs:', OUTPUT_ROOT)
print('Data config:', data_config.to_dict())
print('Training config:', training_config.to_dict())

Dataset: /home/klz/Data/zeronu_benchmark/EXO-200
Outputs: /home/klz/Data/zeronu_benchmark/Transformer_Approach/exo200_detector/results/transformer_official_v1
Data config: {'data_root': '/home/klz/Data/zeronu_benchmark/EXO-200', 'validation_fraction': 0.1, 'test_fraction': 0.2727272727272727, 'baseline_samples': 200, 'classification_amplitude_normalization': True, 'seed': 42, 'input_field': 'Waveforms', 'label_field': 'Charge_cluster_number', 'event_field': 'event_number', 'group_attribute': 'run_number', 'position_field': 'Charge_Clusters_Pos', 'input_shape': [226, 300], 'input_dtype': 'int16', 'num_classes': 2, 'class_names': ['signal', 'background']}
Training config: {'batch_size': 64, 'epochs': 50, 'learning_rate': 0.0005, 'weight_decay': 0.0001, 'gradient_clip_norm': 1.0, 'early_stopping_patience': 5, 'seed': 42, 'num_workers': 0, 'device': 'auto', 'early_stopping_min_delta': 0.0, 'deterministic': False, 'use_amp': False, 'amp_precision': 'auto'}


In [3]:
import os

os.environ["EXO200_RUN_IDS"] = (
    "classification__raw_patches__rope,"
    "classification__segment_summary__rope,"
    "classification__pulse_entities__rope"
)

print(os.environ["EXO200_RUN_IDS"])

EXPERIMENTS = [
    {'tokenization': tokenization, 'position_encoding': position_encoding}
    for tokenization in ('raw_patches', 'segment_summary', 'pulse_entities')
    for position_encoding in ('coordinate_mlp', 'fourier_coordinates', 'rope')
]
ALL_RUN_IDS = {
    f'classification__{row["tokenization"]}__{row["position_encoding"]}'
    for row in EXPERIMENTS
}
requested = os.environ.get('EXO200_RUN_IDS', '').strip()
SELECTED_RUN_IDS = (
    {value.strip() for value in requested.split(',') if value.strip()}
    if requested else set()
)
unknown = SELECTED_RUN_IDS - ALL_RUN_IDS
if unknown:
    raise ValueError(f'Unknown EXO200_RUN_IDS: {sorted(unknown)}')

def make_tokenization_config(name):
    return TokenizationConfig(
        tokenization=name,
        channel_regions=7,
        time_regions=12,
        uniform_entity_fraction=0.5,
        entity_context_size=9,
    )

def run_is_complete(run_dir):
    required = (
        'run_config.json', 'best.pt', 'history.json',
        'metrics.json', 'predictions.npz', 'run_summary.json',
    )
    return all((run_dir / name).is_file() for name in required)

def reset_training_shuffle(data, seed):
    # EXOBench wraps its DataLoader in a prefetch adapter. Resetting the
    # loader-owned generator gives every representation the same epoch-1
    # shuffle sequence while keeping the dataset prepared only once.
    loader = getattr(data.train_loader, 'loader', data.train_loader)
    generator = getattr(loader, 'generator', None)
    if generator is None:
        raise RuntimeError('EXOBench training loader has no seeded generator')
    generator.manual_seed(seed)

def upsert_summary(row):
    if SUMMARY_PATH.is_file():
        table = pd.read_csv(SUMMARY_PATH)
        table = table.loc[table['run_id'] != row['run_id']]
    else:
        table = pd.DataFrame()
    table = pd.concat([table, pd.DataFrame([row])], ignore_index=True)
    table = table.sort_values(['tokenization', 'position_encoding'])
    table.to_csv(SUMMARY_PATH, index=False)

experiment_table = pd.DataFrame([
    {
        'run_id': f'classification__{row["tokenization"]}__{row["position_encoding"]}',
        **row,
    }
    for row in EXPERIMENTS
])
display(experiment_table)
print('Selected runs:', sorted(SELECTED_RUN_IDS) if SELECTED_RUN_IDS else 'all')

classification__raw_patches__rope,classification__segment_summary__rope,classification__pulse_entities__rope


,run_id,tokenization,position_encoding
0,classification__raw_patches__coordinate_mlp,raw_patches,coordinate_mlp
1,classification__raw_patches__fourier_coordinates,raw_patches,fourier_coordinates
2,classification__raw_patches__rope,raw_patches,rope
3,classification__segment_summary__coordinate_mlp,segment_summary,coordinate_mlp
4,classification__segment_summary__fourier_coord...,segment_summary,fourier_coordinates
5,classification__segment_summary__rope,segment_summary,rope
6,classification__pulse_entities__coordinate_mlp,pulse_entities,coordinate_mlp
7,classification__pulse_entities__fourier_coordi...,pulse_entities,fourier_coordinates
8,classification__pulse_entities__rope,pulse_entities,rope


Selected runs: ['classification__pulse_entities__rope', 'classification__raw_patches__rope', 'classification__segment_summary__rope']


In [4]:
print('Preparing the official EXO-200 v1 run-level split once...')
classification_data = prepare_dataset(
    data_config=data_config,
    batch_size=training_config.batch_size,
    num_workers=training_config.num_workers,
)
classification_data.require_two_classes()

split_manifest = validate_split_manifest(
    classification_data, data_root=DATA_ROOT
)
print('Split manifest:', split_manifest['dataset_doi'])

print('Counts:', classification_data.counts)
print('Class counts:', classification_data.class_counts)
print('Runs:', classification_data.runs)
print('Overlap:', classification_data.overlap_counts)

Preparing the official EXO-200 v1 run-level split once...
Split manifest: 10.5281/zenodo.20419164
Counts: {'train': 336632, 'validation': 37987, 'test': 140383}
Class counts: {'train': {'signal': 160803, 'background': 175829}, 'validation': {'signal': 17813, 'background': 20174}, 'test': {'signal': 66062, 'background': 74321}}
Runs: {'train': [8956, 8957, 8958, 8968, 8972, 8999, 9000, 9001, 9008, 9009], 'validation': [8969], 'test': [8967, 8970, 8971, 9010]}
Overlap: {'train_validation': 0, 'train_test': 0, 'validation_test': 0}


In [5]:
for experiment in EXPERIMENTS:
    tokenization = experiment['tokenization']
    position_encoding = experiment['position_encoding']
    run_id = f'classification__{tokenization}__{position_encoding}'
    run_dir = OUTPUT_ROOT / run_id

    if SELECTED_RUN_IDS and run_id not in SELECTED_RUN_IDS:
        continue
    if run_id in COMPLETED_OFFICIAL_RUNS or run_is_complete(run_dir):
        print(f'Skipping completed run: {run_id}')
        continue

    print('\n' + '=' * 88)
    print(run_id)
    print('=' * 88)
    run_dir.mkdir(parents=True, exist_ok=True)
    tokenization_config = make_tokenization_config(tokenization)

    set_seed(training_config.seed, training_config.deterministic)
    reset_training_shuffle(classification_data, training_config.seed)
    model = EXOTransformerClassifier(
        tokenization_config=tokenization_config,
        position_encoding=position_encoding,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
        num_frequencies=6,
        # Frozen a priori from EXO coordinates in [-1, 1]; not tuned on AUC.
        rope_base=math.pi / 2.0,
    )
    parameter_count = sum(
        parameter.numel() for parameter in model.parameters()
        if parameter.requires_grad
    )
    run_config = {
        'run_id': run_id,
        'dataset': 'EXO-200 v1',
        'task': 'classification',
        'data': data_config.to_dict(),
        'counts': classification_data.counts,
        'class_counts': classification_data.class_counts,
        'split_runs': classification_data.runs,
        'split_overlap_counts': classification_data.overlap_counts,
        'training': training_config.to_dict(),
        'representation': model.config_dict(),
        'parameter_count': parameter_count,
    }
    (run_dir / 'run_config.json').write_text(
        json.dumps(run_config, indent=2), encoding='utf-8'
    )
    print('Trainable parameters:', f'{parameter_count:,}')

    training_start = time.perf_counter()
    history = train_model(
        model,
        classification_data.train_loader,
        classification_data.validation_loader,
        config=training_config,
        output_dir=run_dir,
    )
    training_seconds = time.perf_counter() - training_start

    evaluation_start = time.perf_counter()
    metrics = evaluate_model(
        model,
        classification_data.test_loader,
        device=training_config.device,
        output_dir=run_dir,
        use_amp=training_config.use_amp,
        amp_precision=training_config.amp_precision,
    )
    evaluation_seconds = time.perf_counter() - evaluation_start
    checkpoint = torch.load(
        run_dir / 'best.pt', map_location='cpu', weights_only=False
    )
    epochs_completed = len(history)
    row = {
        'run_id': run_id,
        'task': 'classification',
        'tokenization': tokenization,
        'position_encoding': position_encoding,
        'parameter_count': parameter_count,
        'train_events': classification_data.counts['train'],
        'validation_events': classification_data.counts['validation'],
        'test_events': classification_data.counts['test'],
        'epochs_completed': epochs_completed,
        'best_epoch': int(checkpoint['epoch']),
        'best_validation_auc': float(checkpoint['score']),
        'training_seconds': training_seconds,
        'minutes_per_epoch': training_seconds / max(epochs_completed, 1) / 60.0,
        'evaluation_seconds': evaluation_seconds,
        'test_auc': metrics.get('auc'),
        'test_accuracy': metrics.get('accuracy'),
        'test_loss': metrics.get('loss'),
    }
    (run_dir / 'run_summary.json').write_text(
        json.dumps(row, indent=2, allow_nan=True), encoding='utf-8'
    )
    upsert_summary(row)
    print(json.dumps(row, indent=2, allow_nan=True))

Skipping completed run: classification__raw_patches__rope

classification__segment_summary__rope
Trainable parameters: 106,817
Epoch 001/050 | train loss 0.412294 | validation loss 0.325173 | validation macro_auc 0.935529
Epoch 002/050 | train loss 0.332438 | validation loss 0.297885 | validation macro_auc 0.944414
Epoch 003/050 | train loss 0.314906 | validation loss 0.296545 | validation macro_auc 0.948315
Epoch 004/050 | train loss 0.30512 | validation loss 0.300334 | validation macro_auc 0.946533
Epoch 005/050 | train loss 0.297523 | validation loss 0.276723 | validation macro_auc 0.953861
Epoch 006/050 | train loss 0.292241 | validation loss 0.279947 | validation macro_auc 0.950346
Epoch 007/050 | train loss 0.287204 | validation loss 0.263851 | validation macro_auc 0.956058
Epoch 008/050 | train loss 0.283684 | validation loss 0.268397 | validation macro_auc 0.955942
Epoch 009/050 | train loss 0.279478 | validation loss 0.268695 | validation macro_auc 0.955943
Epoch 010/050 | tra

In [6]:
results = pd.read_csv(SUMMARY_PATH) if SUMMARY_PATH.is_file() else pd.DataFrame()
print('Completed official runs:', len(results), '/ 9')
display(results)

Completed official runs: 9 / 9


,run_id,task,tokenization,position_encoding,parameter_count,train_events,validation_events,test_events,epochs_completed,best_epoch,best_validation_auc,training_seconds,minutes_per_epoch,evaluation_seconds,test_auc,test_accuracy,test_loss
0,classification__pulse_entities__coordinate_mlp,classification,pulse_entities,coordinate_mlp,111297,336632,37987,140383,50,46,0.959001,40473.448553,13.491150,336.849847,0.954116,0.883348,0.267769
1,classification__pulse_entities__fourier_coordi...,classification,pulse_entities,fourier_coordinates,115905,336632,37987,140383,49,44,0.962490,78751.861079,26.786347,338.701060,0.959620,0.892437,0.252941
2,classification__pulse_entities__rope,classification,pulse_entities,rope,106689,336632,37987,140383,50,49,0.962209,13078.000281,4.359333,98.574746,0.958720,0.892266,0.254137
3,classification__raw_patches__coordinate_mlp,classification,raw_patches,coordinate_mlp,120769,336632,37987,140383,48,43,0.976098,14150.484076,4.913363,102.582079,0.974123,0.922704,0.201972
4,classification__raw_patches__fourier_coordinates,classification,raw_patches,fourier_coordinates,125377,336632,37987,140383,44,39,0.979691,12413.954687,4.702256,103.787075,0.977804,0.929607,0.184519
5,classification__raw_patches__rope,classification,raw_patches,rope,116161,336632,37987,140383,39,34,0.978195,22319.536988,9.538264,144.915695,0.976599,0.927584,0.187985
6,classification__segment_summary__coordinate_mlp,classification,segment_summary,coordinate_mlp,111425,336632,37987,140383,43,38,0.961995,12256.581500,4.750613,104.615239,0.959483,0.892273,0.253782
7,classification__segment_summary__fourier_coord...,classification,segment_summary,fourier_coordinates,116033,336632,37987,140383,50,49,0.970057,14535.689352,4.845230,103.632963,0.967887,0.906157,0.227152
8,classification__segment_summary__rope,classification,segment_summary,rope,106817,336632,37987,140383,50,49,0.967085,12806.658544,4.268886,96.838195,0.964940,0.901106,0.237772
